In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys
from pathlib import Path

# notebooks/ -> FurnaceMind/
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

In [ ]:
import pandas as pd
from datetime import timedelta

from qdrant_client.models import PointStruct
from utils.helpers import window_id_to_uuid

from utils.settings import settings
from embeddings.sentence_embedding import SentenceEmbedding
from llm.llm_client import OpenRouterClient
from memory.vector_store import QdrantVectorStore
from utils.helpers import build_shift_payload, build_day_payload, build_week_payload, build_biweek_payload

from core.shift_builder import ShiftBuilder
from core.stability_index import FurnaceStabilityIndex
from core.shift_analyzer import ShiftAnalyzer
from core.contextual_analyzer import ContextualAnalyzer
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import json
from pathlib import Path

SHIFT_SCHEMA_PATH = Path(r"C:\All folders\VoptimAlse\FurnaceMind\config\shift_payload_schema.json")
with open(SHIFT_SCHEMA_PATH, "r") as f:
    SHIFT_SCHEMA = json.load(f)

DAY_SCHEMA_PATH = Path(r"C:\All folders\VoptimAlse\FurnaceMind\config\day_payload_schema.json")
with open(DAY_SCHEMA_PATH, "r") as f:
    DAY_SCHEMA = json.load(f)

WEEK_SCHEMA_PATH = Path(r"C:\All folders\VoptimAlse\FurnaceMind\config\weekly_payload_schema.json")
with open(WEEK_SCHEMA_PATH, "r") as f:
    WEEK_SCHEMA = json.load(f)

BIWEEK_SCHEMA_PATH = Path(r"C:\All folders\VoptimAlse\FurnaceMind\config\biweekly_payload_schema.json")
with open(BIWEEK_SCHEMA_PATH, "r") as f:
    BIWEEK_SCHEMA = json.load(f)    

In [ ]:
# Embedding model
embedder = SentenceEmbedding(
    model_name=settings.embedding.model_name,
    device=settings.embedding.device
)

# LLM
llm = OpenRouterClient()

# # Qdrant (supports local + cloud)
# qdrant = QdrantClient(
#     url=settings.qdrant.url,
#     api_key=settings.qdrant.api_key,   
#     timeout=settings.qdrant.timeout
# )

# COLLECTION = settings.qdrant.collection_name

store = QdrantVectorStore()    # calls _ensure_collection internally
qdrant = store.client         # reuse the SAME client
COLLECTION = store.collection_name


In [ ]:
print("QDRANT URL:", settings.qdrant.url)
print("QDRANT API KEY PRESENT:", bool(settings.qdrant.api_key))

In [ ]:
fsi_calculator = FurnaceStabilityIndex(
    critical_parameters=[
        "Process Params - BF2_BODY_ETACO",
        "Process Params - BF2_PROC Top Temp Average",
        "Process Params - BF2_PROC Top Pressure Average",
        "Process Params - coke_rate",
        "Process Params - BF2 CO in BF Gas(%)",
        "Process Params - BF2 CO2 in BF Gas (%)",
        "Process Params - BF2_BODY_PERMEABILITY",
    ],
    primary_kpi="Process Params - BF2_BODY_ETACO",
)

In [ ]:
df_raw = pd.read_csv(
    r"C:\All folders\VoptimAlse\FurnaceMind\data\bf2_data_15min_filtered.csv"
)

In [ ]:
# Example: df_raw already exists
# df_raw columns: ['datetime', 'eta_co', 'top_pressure', ...]
df = df_raw.copy()

df["time (IST)"] = pd.to_datetime(df["time (IST)"])
df = df.sort_values("time (IST)")

# Filter last 2 months
end_date = df["time (IST)"].max()
start_date = end_date - pd.DateOffset(months=2)

df_2m = df[df["time (IST)"] >= start_date].copy()

print(df_2m.shape)

In [ ]:
SHIFT_HOURS = settings.app.shift_hours

df_2m["shift_start"] = (
    df_2m["time (IST)"]
    .dt.floor(f"{SHIFT_HOURS}H")
)

In [ ]:
df_2m["day"] = df_2m["time (IST)"].dt.date
df_2m["week"] = df_2m["time (IST)"].dt.to_period("W").astype(str)
df_2m["bi_week"] = (
    df_2m["time (IST)"]
    .dt.to_period("2W")
    .astype(str)
)

In [ ]:
type(df_2m.index)

In [ ]:
df_2m['time (IST)'] = pd.to_datetime(df_2m['time (IST)'])

# Set index
df_2m = df_2m.set_index('time (IST)')

# REMOVE timezone (critical)
if df_2m.index.tz is not None:
    df_2m.index = df_2m.index.tz_convert(None)

df_2m = df_2m.sort_index()

In [ ]:
print(type(df_2m.index))

In [ ]:
df_2m.shape

In [ ]:
# INITIALIZE COMPONENTS
shift_analyzer = ShiftAnalyzer(settings.anomaly)
shift_builder = ShiftBuilder(settings.app.shift_hours)

# df_2m = your resampled dataframe (already prepared earlier)
shifts = shift_builder.build_shifts(df_2m)

from qdrant_client.models import PointStruct

points = []
shift_payloads = []
prev_shift = None

for shift_id, shift_data in shifts.items():


    # SHIFT DATAFRAME
    df_shift = shift_data.data  


    # LLM SHIFT ANALYSIS
    llm_text, structured = shift_analyzer.analyze(
        shift_data=shift_data,
        prev_shift_data=prev_shift,
        llm=llm
    )


    # FURNACE STABILITY INDEX
    fsi_result = fsi_calculator.compute(
        df=df_shift,
        anomaly_count=structured.get("num_anomalies", 0),
    )

    structured["stability_index"] = fsi_result["stability_index"]
    structured["stability_status"] = fsi_result["stability_status"]
    structured["stability_penalties"] = fsi_result["penalties"]


    # EMBEDDING
    embedding = embedder.embed([llm_text])[0]


    # BUILD SHIFT PAYLOAD
    payload = build_shift_payload(
        shift_data=shift_data,
        structured_summary=structured,
        llm_text=llm_text,
        prev_shift=prev_shift,
        schema=SHIFT_SCHEMA
    )

    payload["shift_id"] = shift_data.shift_id
    payload["start_time"] = shift_data.shift_start.isoformat()
    payload["shift_name"] = shift_data.shift_name

 
    # QDRANT POINT (SHIFT)
    point_id = window_id_to_uuid(shift_data.shift_id)

    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload=payload
        )
    )

    shift_payloads.append(payload)
    prev_shift = shift_data


# SAFE UPSERT (SHIFT LEVEL)
if points:
    qdrant.upsert(collection_name=COLLECTION, points=points)
else:
    print("[INFO] No shift summaries generated — skipping Qdrant upsert")

In [ ]:
from collections import defaultdict
from qdrant_client.models import PointStruct

contextual_analyzer = ContextualAnalyzer(llm)


# GROUP SHIFTS BY DAY
day_groups = defaultdict(list)

for p in shift_payloads:
    start_time = p.get("start_time")
    if not start_time:
        continue
    day_id = start_time[:10]
    day_groups[day_id].append(p)

print("[DEBUG] Day groups:", {k: len(v) for k, v in day_groups.items()})

points = []
day_payloads = []


# BUILD DAY SUMMARIES
for day_id, shifts_in_day in day_groups.items():

    shifts_in_day = sorted(shifts_in_day, key=lambda x: x["start_time"])

    # CORRECT METHOD
    llm_text, structured = contextual_analyzer.build_day_summary(
        day_id=day_id,
        shift_payloads=shifts_in_day
    )


    # AVG STABILITY INDEX
    fsi_values = [
        s["stability_index"]
        for s in shifts_in_day
        if s.get("stability_index") is not None
    ]

    avg_fsi = round(sum(fsi_values) / len(fsi_values), 1) if fsi_values else None


    # STABILITY TREND
    if len(fsi_values) >= 2:
        delta = fsi_values[-1] - fsi_values[0]
        trend = "IMPROVING" if delta > 2 else "DECLINING" if delta < -2 else "STABLE"
    else:
        trend = "UNKNOWN"


    # ENRICH STRUCTURED SUMMARY
    structured["avg_stability_index"] = avg_fsi
    structured["stability_trend"] = trend
    structured["num_shifts"] = len(shifts_in_day)


    # EMBEDDING
    embedding = embedder.embed([llm_text])[0]


    # BUILD DAY PAYLOAD (SCHEMA-SAFE)
    payload = build_day_payload(
        day_id=day_id,
        shift_payloads=shifts_in_day,
        structured_summary=structured,
        llm_text=llm_text,
        schema=DAY_SCHEMA
    )

    point_id = window_id_to_uuid(f"day_{day_id}")

    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload=payload
        )
    )

    day_payloads.append(payload)


# SAFE UPSERT
if points:
    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"[INFO] Upserted {len(points)} day summaries")
else:
    print("[INFO] No day summaries generated — skipping Qdrant upsert")

In [ ]:
from collections import defaultdict
import pandas as pd
from qdrant_client.models import PointStruct

week_groups = defaultdict(list)


# GROUP DAYS BY WEEK
for p in day_payloads:
    day_id = p.get("window_id")   # YYYY-MM-DD
    if not day_id:
        continue

    week_id = str(pd.Period(day_id, freq="W"))
    week_groups[week_id].append(p)

print("[DEBUG] Week groups:", {k: len(v) for k, v in week_groups.items()})

points = []
week_payloads = []


# BUILD WEEK SUMMARIES
for week_id, days in week_groups.items():

    # Sort days chronologically
    days = sorted(days, key=lambda x: x["start_time"])

    # CORRECT ANALYZER METHOD
    llm_text, structured = contextual_analyzer.build_week_summary(
        week_id=week_id,
        day_payloads=days
    )


    # AVG STABILITY INDEX
    fsi_values = [
        d.get("avg_stability_index")
        for d in days
        if d.get("avg_stability_index") is not None
    ]

    avg_fsi = round(sum(fsi_values) / len(fsi_values), 1) if fsi_values else None


    # STABILITY TREND
    if len(fsi_values) >= 2:
        delta = fsi_values[-1] - fsi_values[0]
        trend = "IMPROVING" if delta > 2 else "DECLINING" if delta < -2 else "STABLE"
    else:
        trend = "UNKNOWN"


    # ENRICH STRUCTURED SUMMARY
    structured["avg_stability_index"] = avg_fsi
    structured["stability_trend"] = trend
    structured["num_days"] = len(days)


    # EMBEDDING
    embedding = embedder.embed([llm_text])[0]


    # BUILD WEEK PAYLOAD (SCHEMA-SAFE)
    payload = build_week_payload(
        week_id=week_id,
        day_payloads=days,
        structured_summary=structured,
        llm_text=llm_text,
        schema=WEEK_SCHEMA
    )

    point_id = window_id_to_uuid(f"week_{week_id}")

    # Idempotency guard (optional but recommended)
    if qdrant.retrieve(collection_name=COLLECTION, ids=[point_id]):
        continue

    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload=payload
        )
    )

    week_payloads.append(payload)


# SAFE UPSERT
if points:
    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"[INFO] Upserted {len(points)} week summaries")
else:
    print("[INFO] No week summaries generated — skipping Qdrant upsert")

In [ ]:
from collections import defaultdict
import pandas as pd
from qdrant_client.models import PointStruct

biweek_groups = defaultdict(list)


# GROUP WEEKS BY BI-WEEK
for p in week_payloads:
    week_id = p.get("window_id")
    if not week_id:
        continue

    biweek_id = str(pd.Period(week_id, freq="W").asfreq("2W"))
    biweek_groups[biweek_id].append(p)

print("[DEBUG] Biweek groups:", {k: len(v) for k, v in biweek_groups.items()})

points = []
biweek_payloads = []


# BUILD BI-WEEK SUMMARIES
for biweek_id, weeks in biweek_groups.items():

    # Sort weeks chronologically
    weeks = sorted(weeks, key=lambda x: x["window_id"])

    # CORRECT ANALYZER METHOD
    llm_text, structured = contextual_analyzer.build_biweek_summary(
        biweek_id=biweek_id,
        week_payloads=weeks
    )


    # AVG STABILITY INDEX
    fsi_values = [
        w.get("avg_stability_index")
        for w in weeks
        if w.get("avg_stability_index") is not None
    ]

    avg_fsi = round(sum(fsi_values) / len(fsi_values), 1) if fsi_values else None


    # STABILITY TREND
    if len(fsi_values) >= 2:
        delta = fsi_values[-1] - fsi_values[0]
        trend = "IMPROVING" if delta > 2 else "DECLINING" if delta < -2 else "STABLE"
    else:
        trend = "UNKNOWN"


    # ENRICH STRUCTURED SUMMARY
    structured["avg_stability_index"] = avg_fsi
    structured["stability_trend"] = trend
    structured["num_weeks"] = len(weeks)


    # EMBEDDING
    embedding = embedder.embed([llm_text])[0]

    
    # BUILD BI-WEEK PAYLOAD (SCHEMA-SAFE)
    payload = build_biweek_payload(
        biweek_id=biweek_id,
        week_payloads=weeks,
        structured_summary=structured,
        llm_text=llm_text,
        schema=BIWEEK_SCHEMA
    )

    point_id = window_id_to_uuid(f"biweek_{biweek_id}")

    if qdrant.retrieve(collection_name=COLLECTION, ids=[point_id]):
        continue

    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload=payload
        )
    )

    biweek_payloads.append(payload)


# SAFE UPSERT
if points:
    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"[INFO] Upserted {len(points)} bi-week summaries")
else:
    print("[INFO] No bi-week summaries generated — skipping Qdrant upsert")